In [ ]:
# Import the required libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


In [ ]:
# Load the training and test feature matrices
X_train = pd.read_parquet('../data/X_train.parquet', engine='fastparquet')
X_test = pd.read_parquet('../data/X_test.parquet', engine='fastparquet')

y_train = pd.read_parquet('../data/y_train.parquet', engine='fastparquet')['is_fraud']
y_test = pd.read_parquet('../data/y_test.parquet', engine='fastparquet')['is_fraud']
    
# Verify the dimensions of the datasets
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

In [ ]:
# Scale the features because Logistic Regression is sensitive to different numerical scales
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Initialize Logistic Regression using 'class_weight=balanced' to handle class imbalance
logic_model = LogisticRegression(class_weight='balanced', max_iter=1000)

# Train the baseline model
logic_model.fit(X_train_scaled, y_train)

In [ ]:
# Generamos predicciones en el conjunto de prueba
y_pred = logic_model.predict(X_test_scaled)

# Display the confusion matrix and classification report
print("=== CONFUSION MATRIX (LOGISTIC REGRESSION) ===")
print(confusion_matrix(y_test, y_pred))
print("\n=== CLASSIFICATION REPORT (LOGISTIC REGRESSION) ===")
print(classification_report(y_test, y_pred))

>**Logistic Regression Baseline Evaluation:** Precision and F1-Score for the positive class (fraud) proved insufficient for production needs. Switching strategy to **XGBoost** to capture non-linear decision boundaries.

In [ ]:
# Calculate the class weighting factor for imbalanced classes (Negative Cases / Positive Cases)
class_counts = y_train.value_counts()
scale_weight = class_counts[0] / class_counts[1]

# Initialize XGBoost with class balancing configuration
xgb_model = XGBClassifier(
    n_estimators=300, 
    max_depth=5, 
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)

# Train XGBoost using the original features (feature scaling is not required)
xgb_model.fit(X_train, y_train)

In [ ]:
# Generate standard predictions using the default threshold (0.50)
y_pred_xgb = xgb_model.predict(X_test)

print("=== CONFUSION MATRIX (XGBOOST - THRESHOLD 0.50) ===")
print(confusion_matrix(y_test, y_pred_xgb))
print("\n=== CLASSIFICATION REPORT (XGBOOST - THRESHOLD 0.50) ===")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# Evaluate predictions on the training dataset to measure potential overfitting
y_pred_train = xgb_model.predict(X_train)

print("=== TRAINING SET REPORT (OVERFITTING EVALUATION) ===")
print(classification_report(y_train, y_pred_train))

In [ ]:
# Obtain continuous fraud probabilities (class 1) instead of fixed binary predictions
y_probs = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate using a 0.60 threshold to reduce false positives
y_pred_60 = (y_probs >= 0.60).astype(int)

print("=== THRESHOLD 0.60 ===")
print(confusion_matrix(y_test, y_pred_60))
print(classification_report(y_test, y_pred_60))

In [ ]:
# Evaluate using a 0.70 threshold to optimize precision while maintaining high recall
y_pred_70 = (y_probs >= 0.70).astype(int)

print("=== THRESHOLD 0.70 ===")
print(confusion_matrix(y_test, y_pred_70))
print(classification_report(y_test, y_pred_70))

In [ ]:
# The best performance was achieved with a 0.70 threshold.
# Therefore, cross-validation will preserve this threshold when calculating the F1-score.
# We also evaluate the model independently of the threshold using Average Precision.

# Store performance metrics for each of the 5 validation folds
pr_auc_scores = []
f1_scores_70 = [] 

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in skf.split(X_train, y_train):

# Store performance metrics for each of the 5 validation folds
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train XGBoost on the current fold
    xgb_model.fit(X_tr, y_tr)

    # Obtain fraud probabilities on the validation set
    probs = xgb_model.predict_proba(X_val)[:, 1]

    # Calculate Precision-Recall AUC (independent of the classification threshold)
    pr_auc_scores.append(average_precision_score(y_val, probs))

    # Convert probabilities into binary predictions using the optimized 0.70 threshold
    preds_70 = (probs >= 0.70).astype(int)

    # Calculate F1-score on the validation fold
    score = f1_score(y_val, preds_70)
    f1_scores_70.append(score)

# Display the mean and standard deviation of the cross-validation metrics
print("=== FINAL CROSS-VALIDATION RESULTS (5-FOLD) ===")
print("Mean PR-AUC:", np.mean(pr_auc_scores), "±", np.std(pr_auc_scores))
print("Mean F1-Score (Threshold = 0.70):", np.mean(f1_scores_70), "±", np.std(f1_scores_70))

In [ ]:
# Create a copy of X_test to avoid modifying the original dataframe
df_dashboard = X_test.copy()

# Append ground truth labels
df_dashboard['is_fraud_real'] = y_test.values

# Append continuous fraud probabilities predicted by XGBoost
df_dashboard['fraud_probability'] = y_probs

# Append binary predictions using the optimized 0.70 threshold
df_dashboard['model_prediction'] = y_pred_70

# Append error flag to simplify misclassification tracking (1 = Error, 0 = Correct)
df_dashboard['prediction_error'] = (df_dashboard['is_fraud_real'] != df_dashboard['model_prediction']).astype(int)

# Export consolidated dataframe for Power BI dashboarding
df_dashboard.to_csv('../data/fraud_results_powerbi.csv', index=False)

print("File 'fraud_results_powerbi.csv' generated successfully!")
print(f"Total records exported for dashboarding: {len(df_dashboard)}")

### Model Justification & Decision Strategy

1. **Logistic Regression Baseline:**  
   Used as a benchmark model. Although `class_weight='balanced'` improved fraud detection, the model generated an unacceptable number of false positives (low precision), which in a production environment could unnecessarily block legitimate customer transactions.

2. **XGBoost & Gradient Boosting Advantages:**  
   XGBoost captures complex non-linear relationships between features, such as combinations of transaction time, geographic distance, and transaction amount. The `scale_pos_weight` parameter directly addresses the severe class imbalance.

3. **Decision Threshold Optimization:**  
   Standard classifiers use a default probability threshold of 0.50. Increasing this threshold to **0.70** reduces false alarms while maintaining strong recall for fraudulent transactions.

4. **Stratified K-Fold Cross-Validation:**  
   Due to the extreme class imbalance (~0.52% positive cases), standard cross-validation may produce inconsistent class distributions across folds. `StratifiedKFold` with 5 splits ensures reliable evaluation using **PR-AUC** and **F1-Score** metrics.